In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [3]:
from sqlitesearch import VectorSearchIndex

vs_index = VectorSearchIndex(
    keyword_fields=['course'],
    mode='ivf',
    db_path='faq_vectors2.db'
)

In [5]:
query='I just found out about the program, can I still sign up?'
query_vector = model.encode(query)

results = vs_index.search(
    query_vector,
    filter_dict={'course': 'llm-zoomcamp'},
    num_results=5
)

In [6]:
results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': 'dbf5369006',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Can I use Bluesky for learning in public credits?',
  'answer': 'Yes. Bluesky posts can be used for learn

In [7]:
from rag_helper import RAGBase

class RAGVector(RAGBase):

    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {'course': self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )

In [10]:
from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv()

gemini_client = genai.Client(
    http_options=types.HttpOptions(
        retry_options=types.HttpRetryOptions(
            initial_delay=20.0, 
            attempts=3          
        )
    )
) # picks up the API key from the env variable GEMINI_API_KEY


In [11]:
vector_assistant = RAGVector(
    embedder=model,
    index=vs_index,
    llm_client=gemini_client
)

In [12]:
vector_assistant.rag('the program has already begun, can I still sign up?')

'Yes, you can still join. However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted. \n\nRegistration is not strictly checked, so you can start learning and submitting homework as long as the submission forms are still open.'

In [13]:
vs_index.close()